# Notebook Ingestion Debug

Interactive debug notebook for `utils.ingest.ingest_ipynb_document`.


In [5]:
from pathlib import Path
import json
import base64
import io

from PIL import Image as PILImage

from utils.ingest import ingest_ipynb_document, ingest_document
from utils.summarize import summarize_objects


In [6]:
# Choose notebook to test
NOTEBOOK_PATH = Path(
    "documents/analysis: Interpretable Models - Full Images - Predictions- TFM.ipynb"
)
TOKENIZER_PATH = "local_tokenizer/embeddinggemma"
SHOW = 3

assert NOTEBOOK_PATH.exists(), f"Notebook not found: {NOTEBOOK_PATH}"
print("Testing notebook:", NOTEBOOK_PATH)


Testing notebook: documents/analysis: Interpretable Models - Full Images - Predictions- TFM.ipynb


In [7]:
# Run direct ingestion
texts, tables, images = ingest_ipynb_document(
    str(NOTEBOOK_PATH),
    tokenizer_model_path=TOKENIZER_PATH,
)

print(
    "ingest_ipynb_document counts ->",
    f"texts={len(texts)}",
    f"tables={len(tables)}",
    f"images={len(images)}",
)


Token indices sequence length is longer than the specified maximum sequence length for this model (9531 > 2048). Running this sequence through the model will result in indexing errors


ingest_ipynb_document counts -> texts=67 tables=0 images=6


In [8]:
# Validate dispatcher parity
d_texts, d_tables, d_images = ingest_document(
    str(NOTEBOOK_PATH),
    tokenizer_model_path=TOKENIZER_PATH,
)

print(
    "ingest_document counts ->",
    f"texts={len(d_texts)}",
    f"tables={len(d_tables)}",
    f"images={len(d_images)}",
)


Token indices sequence length is longer than the specified maximum sequence length for this model (9531 > 2048). Running this sequence through the model will result in indexing errors


ingest_document counts -> texts=67 tables=0 images=6


In [9]:
def clip(value, max_len=250):
    value = value or ""
    value = value.replace("", "\n")
    return value if len(value) <= max_len else value[:max_len] + "..."


def safe_json(value):
    try:
        return json.dumps(value, indent=2, default=str)
    except Exception:
        return str(value)


def image_debug_meta(image_b64):
    try:
        raw = base64.b64decode(image_b64)
        with PILImage.open(io.BytesIO(raw)) as img:
            return {"format": img.format, "mode": img.mode, "size": img.size}
    except Exception as exc:
        return {"error": str(exc)}


In [10]:
# Inspect text chunks
for i, obj in enumerate(texts[SHOW:-1]):
    print(f"\n[text {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print("content:", getattr(obj, "text", ""))



[text 0]
metadata: {
  "filename": "documents/analysis: Interpretable Models - Full Images - Predictions- TFM.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 3,
  "cell_type": "code",
  "output_index": null,
  "mime_type": null,
  "sequence_index": 3,
  "chunk_index": 0
}
content: [code cell 3]
def _process_outputs_processed(outputs_processed):
    outputs_processed['prediction_label'] = outputs_processed.prediction_value.apply(lambda x : 0 if x < 0.5 else 1)
    outputs_processed['prediction_label_name'] = outputs_processed.prediction_label.apply(lambda x :  "Clase Control" if x == 0 else "Clase Paciente")
    
    X_columns = list(set(outputs_processed.columns).difference({"label", 'prediction_x', 'prediction_y', 'key', 'label_category', 'prediction_value', 'train_test' , "prediction_label", "prediction_label_name"}))
    y_columns = 'prediction_y'
    y_labels = 'label_category'

    # modify X columns
    X_columns = list(set(X_columns) - set({'HOG_feat

In [11]:
# Inspect table objects
for i, obj in enumerate(tables[:SHOW]):
    print(f"\n[table {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print(
        "markdown:",
        getattr(obj, "markdown", ""),
        print("context:", getattr(obj, "context", "")),
    )


In [12]:
# Inspect image objects
for i, obj in enumerate(images[:SHOW]):
    print(f"\n[image {i}]")
    print("metadata:", safe_json(getattr(obj, "metadata", {})))
    print("context:", (getattr(obj, "context", "")))
    print("decoded_image:", safe_json(image_debug_meta(getattr(obj, "base64", ""))))



[image 0]
metadata: {
  "filename": "documents/analysis: Interpretable Models - Full Images - Predictions- TFM.ipynb",
  "origin": "ipynb",
  "pages": [],
  "bboxes": [],
  "cell_index": 9,
  "cell_type": "code",
  "output_index": 1,
  "mime_type": "image/png",
  "sequence_index": 12
}
context: [output stream]
Report on: linear_regression using main threshold as 0.48
Train data: ---------
	Accuracy: 0.800
	Precision: 0.861
	Recall: 0.715
	F1: 0.782
	Specificity: 0.885

	AUC: 0.843
Test data: ---------
	Accuracy: 0.901
	Precision: 0.765
	Recall: 0.867
	F1: 0.812
	Specificity: 0.912

	AUC: 0.843
[output text]
<Figure size 1200x900 with 1 Axes>

Output image from code cell 9. Code context: linear_regression = InterpretableModel("linear_regression", language = "spanish")
linear_regression.initiate_data(outputs_processed, X_columns, y_columns, y_labels)
linear_regression.fit()
linear_regression.predict()
linear_regression.evaluate()
linear_regression.wirte_report()

[markdown cell 10]
### 

In [13]:
# Optional: run summarization (requires local Ollama model)
RUN_SUMMARIZATION = True
MODEL_NAME = "gemma3:12b"

if RUN_SUMMARIZATION:
    s_texts, s_images, s_tables = summarize_objects(
        texts, images, tables, model_name=MODEL_NAME
    )
    print("summaries generated")
    if s_texts:
        print("text summary sample:", s_texts[0].description)
    if s_tables:
        print("table summary sample:", s_tables[0].description)
    if s_images:
        print("image summary sample:", s_images[0].description)
else:
    print("Skipping summarization. Set RUN_SUMMARIZATION=True to enable.")


100%|██████████| 6/6 [13:03<00:00, 130.52s/it]

summaries generated
text summary sample: This notebook aims to train and evaluate several interpretable models using the outputs of a pre-existing CNN to determine the most useful interpretable model.
image summary sample: The image displays a chart presenting the results of a linear regression model, including accuracy, precision, recall, F1 score, specificity, and AUC for both training and test datasets. The chart is labeled with performance metrics and threshold information.
